# Installing important libraries

In [ ]:
!pip install  openai langchain faiss-cpu pypdf tiktoken docarray PyPDF tiktoken

In [ ]:
!pip install langchain-openai

In [ ]:
%pip install -qU  flashrank

In [4]:
!pip install -qU langchain-community pypdf pillow


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from dotenv import load_dotenv
import os
import shutil
import time
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS


In [6]:
load_dotenv()

True

In [7]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [8]:
# Load PDF documents with error handling
print("Loading PDF documents from ./Policy+Documents...")
try:
    pdf_directory_loader = PyPDFDirectoryLoader("./Policy+Documents")
    documents = pdf_directory_loader.load()
    print(f"✓ Successfully loaded {len(documents)} documents")
    print(f"✓ Total pages: {sum(doc.metadata.get('total_pages', 1) for doc in documents)}")
except Exception as e:
    print(f"Error loading documents: {str(e)}")
    raise

Loading PDF documents from ./Policy+Documents...
✓ Successfully loaded 217 documents
✓ Total pages: 7209
✓ Successfully loaded 217 documents
✓ Total pages: 7209


In [9]:
documents[0].page_content[:100]

'Part A \n<<Date>> \n<<Policyholder’s Name>>  \n<<Policyholder’s Address>> \n<<Policyholder’s Contact Num'

In [10]:
# Split documents into chunks
print("Splitting documents into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
splits = text_splitter.split_documents(documents)
print(f"✓ Created {len(splits)} document chunks")
print(f"✓ Average chunk size: {sum(len(s.page_content) for s in splits) // len(splits)} characters")

Splitting documents into chunks...
✓ Created 760 document chunks
✓ Average chunk size: 848 characters


In [11]:
print(splits[0])

page_content='Part A 
<<Date>> 
<<Policyholder’s Name>>  
<<Policyholder’s Address>> 
<<Policyholder’s Contact Number>> 
 
Dear <<Policyholder’s Name>>,  
 
Sub: Your Policy no. <<  >> 
We are glad to inform you that your proposal has been accepted and the HDFC Life Easy Health (“Policy”) 
being this document, has been issued. We have made every effort to design your Policy in a simple format. We 
have highlighted items of importance so that you may recognize them easily. 
 
Policy document: 
As an evidence of the insurance contract between HDFC Life Insurance Company Limited and you, the Policy 
is enclosed herewith. Please preserve this document safely and also inform your nominees about the same. A 
copy of your proposal form and other relevant documents submitted by you is also enclosed for your 
information and record.  
 
Cancellation in the Free-Look Period: 
 
<< In case you are not agreeable to any of the terms and conditions stated in the Policy, you have the option to' metad

In [12]:
# Initialize embeddings model with timeout and retry settings
print("Initializing OpenAI embeddings model...")
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",  # Using smaller, faster model
    request_timeout=60,  # 60 second timeout
    max_retries=3  # Retry up to 3 times on failure
)
print("✓ Embeddings model initialized")

Initializing OpenAI embeddings model...
✓ Embeddings model initialized
✓ Embeddings model initialized


In [13]:
# Test embedding on a single document
print("Testing embeddings on a sample chunk...")
try:
    test_embedding = embeddings_model.embed_documents([splits[0].page_content])
    print(f"✓ Test embedding successful - dimension: {len(test_embedding[0])}")
except Exception as e:
    print(f"✗ Error testing embeddings: {str(e)}")
    raise

Testing embeddings on a sample chunk...
✓ Test embedding successful - dimension: 1536
✓ Test embedding successful - dimension: 1536


In [14]:
# store = LocalFileStore("./cache/") 

# cached_embedder = CacheBackedEmbeddings.from_bytes_store(
#     embeddings_model,
#     store,
#     namespace="semantic-spotter"
# )

In [16]:
# Preview first few splits (safe check)
print("Preview of document splits:")
try:
    for i, split in enumerate(splits[:3]):
        print(f"\n--- Split {i+1} ---")
        print(f"Source: {split.metadata.get('source', 'Unknown')}")
        print(f"Page: {split.metadata.get('page', 'Unknown')}")
        print(f"Content preview: {split.page_content[:150]}...")
    print(f"\n✓ Total splits available: {len(splits)}")
except Exception as e:
    print(f"Error previewing splits: {e}")
    raise

Preview of document splits:

--- Split 1 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: 0
Content preview: Part A 
<<Date>> 
<<Policyholder’s Name>>  
<<Policyholder’s Address>> 
<<Policyholder’s Contact Number>> 
 
Dear <<Policyholder’s Name>>,  
 
Sub: Yo...

--- Split 2 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: 0
Content preview: information and record.  
 
Cancellation in the Free-Look Period: 
 
<< In case you are not agreeable to any of the terms and conditions stated in the...

--- Split 3 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: 0
Content preview: option to return the Policy to us for cancellation stating the reasons thereof, within 15 days from the date of 
receipt of the Policy. On receipt of ...

✓ Total splits available: 760


In [17]:
# SAFE APPROACH: Use FAISS first (in-memory, more stable) then optionally save to Chroma
# This avoids ChromaDB initialization issues on Windows
import gc
import pickle

def create_vector_store_faiss(splits, embeddings_model, save_path="./faiss_store"):
    """
    Create vector store using FAISS (in-memory, more stable than ChromaDB on Windows).
    This should not crash the kernel.
    """
    print(f"{'='*60}")
    print("Creating vector store using FAISS (in-memory)...")
    print(f"Total document chunks: {len(splits)}")
    print(f"{'='*60}\n")
    
    start_time = time.time()
    
    try:
        # Step 1: Test embeddings first
        print("Step 1: Testing embeddings on first document...")
        try:
            test_embedding = embeddings_model.embed_documents([splits[0].page_content])
            print(f"✓ Test embedding successful (dimension: {len(test_embedding[0])})")
            gc.collect()
        except Exception as e:
            print(f"✗ Error testing embeddings: {e}")
            raise
        
        # Step 2: Create FAISS vector store (this is in-memory, should be safe)
        print("\nStep 2: Creating FAISS vector store...")
        print("Processing documents in small batches to prevent memory issues...\n")
        
        # Process in small batches to avoid memory issues
        batch_size = 10
        all_docs = []
        
        for i in range(0, len(splits), batch_size):
            batch = splits[i:i+batch_size]
            batch_num = (i // batch_size) + 1
            total_batches = (len(splits) + batch_size - 1) // batch_size
            
            print(f"Processing batch {batch_num}/{total_batches} ({len(batch)} documents)...", end=" ", flush=True)
            
            try:
                # Embed the batch
                texts = [doc.page_content for doc in batch]
                embeddings = embeddings_model.embed_documents(texts)
                
                # Store for FAISS creation
                all_docs.extend(batch)
                
                print("✓")
                
                # Small delay and garbage collection
                if batch_num < total_batches:
                    gc.collect()
                    time.sleep(0.5)
                    
            except Exception as e:
                print(f"✗ Error in batch {batch_num}: {e}")
                print("Continuing with next batch...")
                continue
        
        # Step 3: Create FAISS vector store from all documents
        print(f"\nStep 3: Creating FAISS index from {len(all_docs)} documents...")
        try:
            vectordb = FAISS.from_documents(
                documents=all_docs,
                embedding=embeddings_model
            )
            print("✓ FAISS vector store created successfully")
        except Exception as e:
            print(f"✗ Error creating FAISS store: {e}")
            raise
        
        # Step 4: Save to disk (PERSISTENT STORAGE)
        print(f"\nStep 4: Saving vector store to disk (persistent storage)...")
        try:
            os.makedirs(os.path.dirname(save_path) if os.path.dirname(save_path) else ".", exist_ok=True)
            vectordb.save_local(save_path)
            print(f"✓ Vector store saved to: {save_path}")
            print(f"✓ Vector store is PERSISTENT - it will survive kernel restarts!")
            print(f"✓ To load it later, use: load_saved_vector_store('{save_path}')")
            
            # Verify the save
            if os.path.exists(save_path):
                file_count = len(os.listdir(save_path))
                print(f"✓ Verification: {file_count} files saved to disk")
        except Exception as e:
            print(f"⚠ Warning: Could not save to disk: {e}")
            print("Vector store is in memory only (will be lost when kernel restarts)")
        
        elapsed_time = time.time() - start_time
        print(f"\n{'='*60}")
        print(f"✓ Successfully created FAISS vector store!")
        print(f"✓ Total documents processed: {len(all_docs)}")
        print(f"✓ Time taken: {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)")
        print(f"{'='*60}\n")
        
        return vectordb
        
    except Exception as e:
        print(f"\n{'='*60}")
        print(f"✗ Error: {e}")
        print(f"Error type: {type(e).__name__}")
        print(f"{'='*60}")
        raise

# Try FAISS approach (more stable on Windows)
print("Creating vector store using FAISS (in-memory approach)...")
print("This should be more stable and prevent kernel crashes.\n")

try:
    vectordb = create_vector_store_faiss(splits, embeddings_model, "./faiss_store")
    print("✓ Vector store creation completed successfully!")
    print("\nNote: FAISS vector store is ready. You can use it for similarity search.")
except Exception as e:
    print(f"\n{'='*60}")
    print("ERROR: FAISS method also failed")
    print(f"Error: {str(e)}")
    print(f"Error type: {type(e).__name__}")
    print(f"{'='*60}")
    print("\nThis suggests the issue might be with:")
    print("1. OpenAI API connection/authentication")
    print("2. Memory issues on your system")
    print("3. Python environment issues")
    print("\nTry:")
    print("1. Restart the kernel completely")
    print("2. Check OpenAI API key: print(os.getenv('OPENAI_API_KEY')[:10] + '...')")
    print("3. Test embeddings manually: embeddings_model.embed_documents(['test'])")
    raise

Creating vector store using FAISS (in-memory approach)...
This should be more stable and prevent kernel crashes.

Creating vector store using FAISS (in-memory)...
Total document chunks: 760

Step 1: Testing embeddings on first document...
✓ Test embedding successful (dimension: 1536)

Step 2: Creating FAISS vector store...
Processing documents in small batches to prevent memory issues...

Processing batch 1/76 (10 documents)... ✓ Test embedding successful (dimension: 1536)

Step 2: Creating FAISS vector store...
Processing documents in small batches to prevent memory issues...

Processing batch 1/76 (10 documents)... ✓
✓
Processing batch 2/76 (10 documents)... Processing batch 2/76 (10 documents)... ✓
✓
Processing batch 3/76 (10 documents)... Processing batch 3/76 (10 documents)... ✓
✓
Processing batch 4/76 (10 documents)... Processing batch 4/76 (10 documents)... ✓
✓
Processing batch 5/76 (10 documents)... Processing batch 5/76 (10 documents)... ✓
✓
Processing batch 6/76 (10 documents

In [21]:
# Function to load existing vector store (FAISS only)
def load_vector_store(store_path="./faiss_store", store_type="faiss"):
    """
    Load an existing vector store from disk.
    Only FAISS stores are supported in this notebook.
    """
    try:
        if store_type.lower() == "faiss":
            if not os.path.exists(store_path):
                print(f"✗ FAISS vector store not found at: {store_path}")
                return None

            vectordb = FAISS.load_local(
                folder_path=store_path,
                embeddings=embeddings_model,
                allow_dangerous_deserialization=True,
            )
            print(f"✓ Successfully loaded FAISS vector store from: {store_path}")
            return vectordb
        else:
            print(f"✗ Unsupported store_type: {store_type}. Only 'faiss' is supported in this notebook.")
            return None
    except Exception as e:
        print(f"✗ Error loading vector store: {str(e)}")
        return None

# Uncomment to load existing vector store:
# vectordb = load_vector_store("./faiss_store", "faiss")


In [22]:
# Install required packages for advanced retrieval
%pip install -q langchain-community sentence-transformers


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
vectordb = load_vector_store()

✓ Successfully loaded FAISS vector store from: ./faiss_store


In [25]:
vectordb.as_retriever()

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000287D8FDA150>, search_kwargs={})

In [26]:
from langchain.tools import tool
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

compressor = FlashrankRerank()
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=vectordb.as_retriever(search_kwargs={"k": 20})
)

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query"""
    retrieved_docs = compression_retriever.invoke(
    query
)
    # retrieved_docs = vectordb.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n Page Content: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [27]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

# Instantiate the LLM
llm = ChatOpenAI(model_name="gpt-4o-mini", streaming=True)

tools = [retrieve_context]

prompt = """You are an insurance policy QA assistant. Answer questions ONLY based on the policy documents provided in details.

CRITICAL OUTPUT FORMAT RULES:
1. ALWAYS respond with EXACTLY this format, nothing else:
answer: [your answer here]
source: [metadata info from retrieved document pdf name and page no.]
2. Do NOT include any explanation, preamble, or extra text
3. Do NOT repeat the question
4. Do NOT include metadata (producer, creator, page, author, etc.)
5. Use only the actual policy content
6. If answer not found, write: "Not found in the provided policy context."

EXAMPLE OUTPUT:
answer: The policy covers up to $500,000 for life insurance
"""

prompt = """You are an insurance policy QA assistant. Answer questions ONLY based on the policy documents provided in details.

CRITICAL OUTPUT FORMAT RULES:
1. ALWAYS respond with EXACTLY this format, nothing else:
answer: [your answer here]
source: [metadata info from retrieved document pdf name and page no.]

EXAMPLE OUTPUT:
answer: The policy covers up to $500,000 for life insurance
"""

agent = create_agent(llm, tools, system_prompt=prompt)


In [29]:
from langchain.messages import HumanMessage

def insurance_agent(query: str):
    response = agent.invoke({
        'messages': [
            HumanMessage(content=(
            query
            ))
        ]
    })

    print(response['messages'][-1].content)

In [30]:
insurance_agent( "What is the life insurance policy coverage amount?")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The maximum benefit payable by the Company on Accidental Death of an Insured Member under all policies is limited to a total of Rs. 10,000,000 (Rupees 1 crore only). The specific Sum Assured for each Scheme Member may vary and will be detailed in the Certificate of Insurance. 
source: HDFC Life Group Poorna Suraksha (101N137V02) - Policy Document, page 6


In [31]:
insurance_agent( "Can a 100 year plus person do a term insurance?")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: There is no specific mention in the policy documents about the eligibility of a person over 100 years old for term insurance. However, generally, insurance providers may have age limits for issuing new term insurance policies, typically considering individuals under a certain age, such as 60 or 65, but this varies by provider. It’s advised to consult with the specific insurer for their criteria.

source: Policy Documents\HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf, page 16


In [32]:
insurance_agent("what is the Definitions of Critical Illnesses? based on policy?")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The policy includes definitions for covered Critical Illnesses such as Myocardial Infarction, which involves the first occurrence of heart attack of specific severity, evidenced by clinical symptoms, electrocardiogram changes, and elevation of specific enzymes. Exclusions include other acute coronary syndromes and angina pectoris, among others.
source: HDFC Life Group Poorna Suraksha (101N137V02) - Policy Document, page 27.


In [33]:
# retrieve_context("what is the Definitions of Critical Illnesses? based on policy?")

In [34]:
# retrieve_context("Can a 100 year plus person do a term insurance?")

In [35]:
insurance_agent("what is the life insurance coverage for disability?")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The document does not specifically mention any life insurance coverage for disability. It primarily discusses coverages related to death benefits and other options, but no details on disability coverage are provided. 
source: HDFC Life Group Poorna Suraksha (101N137V02) - Policy Document.pdf, page 8
